# ME 566 Homework 2

Add your camera images to `images/` using the names in `images/README.md`, then choose **Run All**.


Shared setup (imports and image paths)


In [ ]:
from pathlib import Path
from time import perf_counter

import cv2
import numpy as np

from src.detectors import DoG, Fast, Harris, Octree, ShiTomasi
from src.io.image_reader import ImageReader
from src.matchers import Brief, Orb, SIFT
from src.utils import draw_features, save_image

ROOT = Path.cwd()
IMAGE_DIR = ROOT / "images"
OUTPUT_DIR = ROOT / "outputs"
IMAGES = None


def load_images():
    """Read the images directory once using the project's ImageReader."""
    global IMAGES
    if IMAGES is None:
        if not IMAGE_DIR.exists():
            raise FileNotFoundError("Create images/ and add your camera images.")
        IMAGES = ImageReader(str(IMAGE_DIR)).images
        if len(IMAGES) == 0:
            raise FileNotFoundError("No images were found in images/.")
        OUTPUT_DIR.mkdir(exist_ok=True)
        print(f"Loaded {len(IMAGES)} image(s) through ImageReader.")
    return IMAGES


def show_output(image, filename):
    output_path = OUTPUT_DIR / filename
    save_image(str(output_path), image)
    print(f"Saved: {output_path.relative_to(ROOT)}")
    try:
        from IPython.display import Image, display
        display(Image(filename=str(output_path)))
    except ImportError:
        pass


def print_table(rows, columns):
    text_rows = [[str(row[column]) for column in columns] for row in rows]
    widths = [max(len(column), *(len(row[i]) for row in text_rows)) for i, column in enumerate(columns)]
    print("  ".join(column.ljust(widths[i]) for i, column in enumerate(columns)))
    print("  ".join("-" * width for width in widths))
    for row in text_rows:
        print("  ".join(value.ljust(widths[i]) for i, value in enumerate(row)))


1: A comparison of DoG, Harris, Shi-Tomasi, and FAST feature detectors addressing

- Repeatability,
- Speed, and
- Number of features detected.

Make sure to see how they behave with changes in scale, translation, and rotation. Use images
that captured by your camera. Do not digitally rotate or scale the images. This leads to artifacts
that make the test results meaningless

In [ ]:
# Question 1: use the existing detector classes
DETECTORS = {"DoG": DoG, "Harris": Harris, "Shi-Tomasi": ShiTomasi, "FAST": Fast}
Q1_CACHE = None


def _q1_detect_everything():
    """Run each provided detector once on every image from ImageReader."""
    global Q1_CACHE
    if Q1_CACHE is not None:
        return Q1_CACHE
    Q1_CACHE = []
    for image_index, image in enumerate(load_images(), start=1):
        for name, detector_class in DETECTORS.items():
            detector = detector_class(image)
            start = perf_counter()
            keypoints = detector.detect_features()
            Q1_CACHE.append({
                "image": f"image_{image_index}", "detector": name, "features": len(keypoints),
                "time (ms)": round((perf_counter() - start) * 1000, 2),
                "detector object": detector, "keypoints": keypoints,
            })
    return Q1_CACHE


def check_repeatability():
    """Save feature overlays for every detector/image pair for visual repeatability comparison."""
    results = _q1_detect_everything()
    for result in results:
        detector = result["detector object"]
        overlay = draw_features(detector.image, result["keypoints"], color=(0, 255, 0))
        safe_name = result["detector"].lower().replace("-", "_")
        show_output(overlay, f"q1_{result['image']}_{safe_name}.png")
    print("Compare matching physical features across the saved overlays for your scale, translation, and rotation images.")
    return results


def check_speed():
    rows = [{"image": item["image"], "detector": item["detector"], "time (ms)": item["time (ms)"]}
            for item in _q1_detect_everything()]
    print_table(rows, ("image", "detector", "time (ms)"))
    return rows


def check_number_of_features():
    rows = [{"image": item["image"], "detector": item["detector"], "features": item["features"]}
            for item in _q1_detect_everything()]
    print_table(rows, ("image", "detector", "features"))
    return rows


In [ ]:
# Question 1 checks
q1_repeatability = check_repeatability()
q1_speed = check_speed()
q1_feature_count = check_number_of_features()


2: Implement a method for selecting a subset of features based on contemporary research. The
method must provide a reasonable set of features covering the whole image. Citations are needed
for this. Octrees and ANMS are candidates for this. I will provide the ANMS paper on
Blackboard.

In [ ]:
# Question 2: use the existing FAST and Octree detector classes
Q2_CACHE = None


def _q2_feature_sets():
    """Use the first ImageReader image for the spatial-selection example."""
    global Q2_CACHE
    if Q2_CACHE is None:
        image = load_images()[0]
        fast = Fast(image)
        octree = Octree(image)
        Q2_CACHE = {
            "fast": fast, "candidates": fast.detect_features(),
            "octree": octree, "selected": octree.detect_features(max_features=500),
        }
    return Q2_CACHE


def show_candidate_features():
    data = _q2_feature_sets()
    show_output(draw_features(data["fast"].image, data["candidates"], color=(0, 0, 255)), "q2_fast_candidates.png")
    return data["candidates"]


def show_selected_features():
    data = _q2_feature_sets()
    show_output(draw_features(data["octree"].image, data["selected"], color=(0, 255, 255)), "q2_octree_selected.png")
    return data["selected"]


def check_image_coverage():
    data = _q2_feature_sets()
    rows = [{"method": "FAST candidates", "features": len(data["candidates"])},
            {"method": "Octree-selected", "features": len(data["selected"])}]
    print_table(rows, ("method", "features"))
    print("Citation: Rublee et al., 'ORB: an efficient alternative to SIFT or SURF,' ICCV 2011.")
    return rows


In [ ]:
# Question 2 checks
q2_candidates = show_candidate_features()
q2_selected = show_selected_features()
q2_coverage = check_image_coverage()


3: Conduct feature matching using SIFT, BRIEF, and ORB feature descriptors. Make sure to pick
one type of detections scheme from above to ensure a good comparison. Look at the speed and
quality of matching. Produce a confusion matrix for each one of these.

In [ ]:
# Question 3: use one shared FAST detector with the existing descriptor matchers
MATCHERS = {"SIFT": SIFT, "BRIEF": Brief, "ORB": Orb}
Q3_CACHE = None


def _q3_match_everything():
    """Match the first two ImageReader images with the same FAST keypoints."""
    global Q3_CACHE
    if Q3_CACHE is not None:
        return Q3_CACHE
    images = load_images()
    if len(images) < 2:
        raise ValueError("Question 3 needs at least two images in images/.")
    left_image, right_image = images[0], images[1]
    left_keypoints = Fast(left_image).detect_features()
    right_keypoints = Fast(right_image).detect_features()
    Q3_CACHE = []

    for name, matcher_class in MATCHERS.items():
        matcher = matcher_class(left_image)
        start = perf_counter()
        if name == "SIFT":
            matched_image = matcher.match(right_image, ratio=0.7, keypoints1=left_keypoints, keypoints2=right_keypoints)
        else:
            matched_image = matcher.match(right_image, max_matches=50, keypoints1=left_keypoints, keypoints2=right_keypoints)
        Q3_CACHE.append({
            "descriptor": name, "matched image": matched_image, "matches": len(matcher.matches),
            "time (ms)": round((perf_counter() - start) * 1000, 2),
        })
    return Q3_CACHE


def show_feature_matches():
    for result in _q3_match_everything():
        show_output(result["matched image"], f"q3_{result['descriptor'].lower()}_matches.png")


def check_matching_speed():
    rows = [{"descriptor": item["descriptor"], "time (ms)": item["time (ms)"]}
            for item in _q3_match_everything()]
    print_table(rows, ("descriptor", "time (ms)"))
    return rows


def check_matching_quality():
    rows = [{"descriptor": item["descriptor"], "matches": item["matches"]}
            for item in _q3_match_everything()]
    print_table(rows, ("descriptor", "matches"))
    print("Use the saved overlays to identify correct and incorrect matches.")
    return rows


def build_confusion_matrices(labels):
    """Build matrices after manually labeling match outcomes: (predicted_match, actual_match)."""
    matrices = {}
    for name, outcomes in labels.items():
        matrix = np.zeros((2, 2), dtype=int)  # [[TN, FP], [FN, TP]]
        for predicted, actual in outcomes:
            matrix[int(actual), int(predicted)] += 1
        matrices[name] = matrix
        print(f"{name}: [[TN, FP], [FN, TP]] = {matrix.tolist()}")
    return matrices


In [ ]:
# Question 3 checks
show_feature_matches()
q3_speed = check_matching_speed()
q3_quality = check_matching_quality()

# After manually reviewing match overlays, pass your labels to build_confusion_matrices(...).


4: For the matching method of your choice, construct a ROC curve. Talk about the idea parameters
for your matching technique.